# Gemma 4 Softmax-to-SBA Homotopy Conversion Ablations
This notebook runs the **ECoT / Distillation SBA Conversion Debugging & Ablation** experiments on a Google Colab A100 runtime.

### Setup Instructions:
1. Upload the files in the flat `sba_math_handoff` folder to your Google Drive inside a folder named **`sba_project`** OR upload them directly to Colab's file panel (the folder icon on the left).
2. Run the cells in order. The notebook will automatically copy files, reconstruct directory structures, mount Drive, and set up Hugging Face credentials.

In [ ]:
# 1. Optional: Mount Google Drive
# If you uploaded your files to Google Drive, run this cell to mount it.
# If you uploaded files directly using Colab's file browser, you can skip this cell.
from google.colab import drive
try:
    drive.mount('/content/drive')
except Exception as e:
    print("Drive mount skipped or failed:", e)

In [ ]:
# 2. Reconstruct workspace and extract checkpoints
import os
import shutil

# Create local workspace
!mkdir -p /content/sba_workspace

# Check if files are in Google Drive
if os.path.exists('/content/drive/MyDrive/sba_project'):
    print("Found sba_project folder in Google Drive. Copying files...")
    !cp -r /content/drive/MyDrive/sba_project/* /content/sba_workspace/
else:
    print("Google Drive folder not found. Checking direct uploads in /content...")
    # Copy files directly uploaded to the root /content folder
    !cp -r /content/*.py /content/*.ipynb /content/*.zip /content/*.json /content/*.jsonl /content/sba_workspace/ 2>/dev/null || true

%cd /content/sba_workspace

# Reconstruct target directories needed by evaluate_mmlu.py and run_softmax_bridge_conversion.py
!mkdir -p data results/gemma4_e4b_sba_distill_bridge_b200/checkpoint-1500

# Move dataset to expected path
if os.path.exists("sba_self_distill.jsonl"):
    shutil.copy2("sba_self_distill.jsonl", "data/sba_self_distill.jsonl")
    print("Dataset moved to data/sba_self_distill.jsonl")

# Unzip checkpoint-1500
if os.path.exists("checkpoint-1500.zip"):
    !unzip -q -o checkpoint-1500.zip -d results/gemma4_e4b_sba_distill_bridge_b200/checkpoint-1500/
    print("Unzipped checkpoint-1500.zip to results/gemma4_e4b_sba_distill_bridge_b200/checkpoint-1500/")

print("Workspace content:")
!ls -la

In [ ]:
# 3. Install required packages
!pip install -q transformers datasets peft accelerate tqdm triton sentencepiece

In [ ]:
# 4. Authenticate with Hugging Face Hub
from huggingface_hub import notebook_login
notebook_login()

## Phase 1: The Timeline Diagnostic
We run MMLU on **`checkpoint-1500`** (which we extracted above) to establish whether the representational damage occurred during transition (Phase 1) or recovery (Phase 2).

In [ ]:
# Run MMLU on checkpoint-1500
!python evaluate_mmlu.py \
    --model_name_or_path google/gemma-4-E4B \
    --checkpoint_dir /content/sba_workspace/results/gemma4_e4b_sba_distill_bridge_b200/checkpoint-1500 \
    --output_file /content/sba_workspace/results/mmlu_checkpoint_1500_diagnostic.json \
    --batch_size 8 \
    --limit_questions 20

### Timeline Diagnostic Criteria:
* **If MMLU <= 67% (Drop of >= 4.23pp)**: Phase 1 contributed significantly to the damage. **All subsequent training ablations must start from Step 0.**
* **If MMLU >= 69% (Drop of <= 2.23pp)**: Phase 1 was relatively safe; damage is Phase 2-dominant. **All subsequent training ablations must resume from `checkpoint-1500`** to save compute.

## Phase 3: The Experimental Sequence
Run these experiments based on the Phase 1 diagnostic result.

### Experiment 1: The EAR Ablation (Highest Priority)
This test disables the Energy-Aware Rescaling (EAR) amplification, which we hypothesize is corrupting Adam's variance tracking buffers.

In [ ]:
# OPTION A: Start from Step 0 (If Phase 1 MMLU <= 67%)
!accelerate launch run_softmax_bridge_conversion.py \
    --model_name_or_path google/gemma-4-E4B \
    --train_file data/sba_self_distill.jsonl \
    --output_dir /content/drive/MyDrive/sba_project/results/exp1_ear_ablation_step0 \
    --max_seq_length 512 \
    --per_device_train_batch_size 4 \
    --gradient_accumulation_steps 8 \
    --stage0_steps 300 \
    --stage1_steps 1000 \
    --stage2_steps 3700 \
    --loss_chunk_size 32 \
    --save_steps 500 \
    --logging_steps 10 \
    --mixed_precision bf16 \
    --orthogonal_projection \
    --disable_ear

In [ ]:
# OPTION B: Resume from Checkpoint-1500 (If Phase 1 MMLU >= 69%)
!accelerate launch run_softmax_bridge_conversion.py \
    --model_name_or_path google/gemma-4-E4B \
    --train_file data/sba_self_distill.jsonl \
    --output_dir /content/drive/MyDrive/sba_project/results/exp1_ear_ablation_resume1500 \
    --max_seq_length 512 \
    --per_device_train_batch_size 4 \
    --gradient_accumulation_steps 8 \
    --stage0_steps 300 \
    --stage1_steps 1000 \
    --stage2_steps 3700 \
    --loss_chunk_size 32 \
    --save_steps 500 \
    --logging_steps 10 \
    --mixed_precision bf16 \
    --orthogonal_projection \
    --disable_ear \
    --resume_from_checkpoint /content/sba_workspace/results/gemma4_e4b_sba_distill_bridge_b200/checkpoint-1500

### Experiment 2: SlimPajama Replay Baseline (Contingent)
If Experiment 1 fails, run training with experience replay: `--replay_file` and `--replay_mixing_ratio 0.1`.

In [ ]:
# OPTION A: Start from Step 0 (with SlimPajama 10% Replay, EAR scaling active)
# Note: SlimPajama file should be uploaded to Google Drive or Colab direct folder as 'slimpajama_sample.jsonl'
!accelerate launch run_softmax_bridge_conversion.py \
    --model_name_or_path google/gemma-4-E4B \
    --train_file data/sba_self_distill.jsonl \
    --output_dir /content/drive/MyDrive/sba_project/results/exp2_replay_step0 \
    --max_seq_length 512 \
    --per_device_train_batch_size 4 \
    --gradient_accumulation_steps 8 \
    --stage0_steps 300 \
    --stage1_steps 1000 \
    --stage2_steps 3700 \
    --loss_chunk_size 32 \
    --save_steps 500 \
    --logging_steps 10 \
    --mixed_precision bf16 \
    --orthogonal_projection \
    --replay_file /content/sba_workspace/slimpajama_sample.jsonl \
    --replay_mixing_ratio 0.1

In [ ]:
# OPTION B: Resume from Checkpoint-1500 (with SlimPajama 10% Replay, EAR scaling active)
!accelerate launch run_softmax_bridge_conversion.py \
    --model_name_or_path google/gemma-4-E4B \
    --train_file data/sba_self_distill.jsonl \
    --output_dir /content/drive/MyDrive/sba_project/results/exp2_replay_resume1500 \
    --max_seq_length 512 \
    --per_device_train_batch_size 4 \
    --gradient_accumulation_steps 8 \
    --stage0_steps 300 \
    --stage1_steps 1000 \
    --stage2_steps 3700 \
    --loss_chunk_size 32 \
    --save_steps 500 \
    --logging_steps 10 \
    --mixed_precision bf16 \
    --orthogonal_projection \
    --resume_from_checkpoint /content/sba_workspace/results/gemma4_e4b_sba_distill_bridge_b200/checkpoint-1500 \
    --replay_file /content/sba_workspace/slimpajama_sample.jsonl \
    --replay_mixing_ratio 0.1

### Experiment 3: The 2x2 Factorial Ablations (Contingent)
If Experiment 2 leaves a gap, execute the following configurations to isolate frozen A vs. trainable A and replay configurations.

In [ ]:
# Run B: Replay 10% + Frozen A
!accelerate launch run_softmax_bridge_conversion.py \
    --model_name_or_path google/gemma-4-E4B \
    --train_file data/sba_self_distill.jsonl \
    --output_dir /content/drive/MyDrive/sba_project/results/exp3_runB \
    --max_seq_length 512 \
    --per_device_train_batch_size 4 \
    --gradient_accumulation_steps 8 \
    --stage0_steps 300 \
    --stage1_steps 1000 \
    --stage2_steps 3700 \
    --loss_chunk_size 32 \
    --save_steps 500 \
    --logging_steps 10 \
    --mixed_precision bf16 \
    --orthogonal_projection \
    --replay_file /content/sba_workspace/slimpajama_sample.jsonl \
    --replay_mixing_ratio 0.1 \
    --freeze_lora_a

In [ ]:
# Run D: Replay 0% + Frozen A
!accelerate launch run_softmax_bridge_conversion.py \
    --model_name_or_path google/gemma-4-E4B \
    --train_file data/sba_self_distill.jsonl \
    --output_dir /content/drive/MyDrive/sba_project/results/exp3_runD \
    --max_seq_length 512 \
    --per_device_train_batch_size 4 \
    --gradient_accumulation_steps 8 \
    --stage0_steps 300 \
    --stage1_steps 1000 \
    --stage2_steps 3700 \
    --loss_chunk_size 32 \
    --save_steps 500 \
    --logging_steps 10 \
    --mixed_precision bf16 \
    --orthogonal_projection \
    --freeze_lora_a

## Phase 4: Dual-Gate Success Criteria Checks
Run evaluations on your resulting target checkpoint to check if both gates are passed.

In [ ]:
# Gate 1: Check MMLU Accuracy (should be >= 68%)
# Replace the checkpoint_dir with the path to the best checkpoint of your experiment run
!python evaluate_mmlu.py \
    --model_name_or_path google/gemma-4-E4B \
    --checkpoint_dir /content/drive/MyDrive/sba_project/results/exp1_ear_ablation_step0/checkpoint-5000 \
    --output_file /content/drive/MyDrive/sba_project/results/mmlu_ablation_final.json \
    --batch_size 8

In [ ]:
# Gate 2: Check SBA Bipolar Activation metrics (Negative Fraction >= 8%, BOS share < 5%)
!python gemma4_diagnostics_v4.py